# PolyChain: Polymer Property Prediction
**AISEHack 2.0 - Polymer Property Prediction**

Run all cells in order. Autosave: completed folds are skipped on re-run.
Supports resume after disconnection.

## Pipeline
1. Install & setup
2. Clone repo (feat/competition-data-adaptation branch)
3. Build features + generate CV splits
4. Train models via scheduler (GPU: polychain, gcn, gat, fusionnet; CPU: ridge, xgb, lgb, catboost, rf, mlp)
5. Ensemble: stacking + weighted blend
6. Export final submission.csv

In [ ]:
# Cell 1: Install dependencies
import sys, os, json, pickle, subprocess, shutil, time, warnings, zipfile
from pathlib import Path

print("Python:", sys.version[:6])
print("Executable:", sys.executable)

# Check available package managers
conda_candidates = [
    "/opt/conda/bin/conda",
    "/usr/local/conda/bin/conda",
    shutil.which("conda"),
]
conda = None
for c in conda_candidates:
    if c and Path(c).exists():
        conda = c
        print(f"Found conda: {c}")
        break
else:
    print("conda not found, will use pip")

try:
    import rdkit
    print("rdkit already installed")
except ImportError:
    print("Installing rdkit...")
    if conda:
        r = subprocess.run(
            [conda, "install", "-c", "conda-forge", "rdkit", "-y", "-q"],
            capture_output=True, text=True, timeout=300
        )
        print(r.stdout[-300:] if r.stdout else "")
        print(r.stderr[-300:] if r.stderr else "")
    else:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "rdkit"],
            capture_output=True, text=True, timeout=120
        )
        print(r.stdout[-300:] if r.stdout else "")
        print(r.stderr[-300:] if r.stderr else "")
    import rdkit
    print("rdkit ready!")

try:
    import matplotlib
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "matplotlib", "seaborn"],
        timeout=120
    )

# GPU detection and PyTorch install (reusable module)
from setup_kaggle import detect_gpu, install_torch, configure_cudnn
gpu_type = detect_gpu()
print(f"Detected GPU: {gpu_type}")
install_torch(gpu_type)

# Verify via subprocess (avoids kernel import cache)
v = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True, timeout=30
)
print("Verified:", v.stdout.strip() if v.returncode == 0 else v.stderr.strip())

# Install PyTorch Geometric (for GNN models: gcn, gat, polychain)
try:
    import torch_geometric
    print("PyG already installed")
except ImportError:
    print("Installing PyTorch Geometric...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "torch_geometric"], capture_output=True, timeout=120)
    import torch_geometric
    print("PyG ready!")

configure_cudnn()
print("Dependencies ready")


In [ ]:
# Cell 2: Git clone/pull feat/competition-data-adaptation
import subprocess, os

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/NotShubham1112/Poly.git"
BRANCH = "feat/competition-data-adaptation"

os.chdir(WORK_DIR)
# Clone or pull
if not os.path.exists(os.path.join(WORK_DIR, "polymer_competition")):
    subprocess.check_call([
        "git", "clone", "-b", BRANCH, "--single-branch", REPO_URL
    ])
# Copy contents to WORK_DIR for direct import
subprocess.check_call(
    ["cp", "-r", "Poly/polymer_competition/.", "."], shell=True
)
print("Repository cloned and copied to WORK_DIR")


In [ ]:
# Cell 3: Feature build + CV splits generation
!python -m features.build_features
!python -m training.splits --data data/train.csv
print("Features built and CV splits generated")


In [ ]:
# Cell 4: Scheduler - train all models
# GPU models: polychain, gcn, gat, fusionnet
# CPU models: ridge, xgb, lgb, catboost, rf, mlp
!python -m training.scheduler --targets tg egc --models ridge xgb lgb catboost rf mlp gcn gat polychain
print("All models trained via scheduler")


In [ ]:
# Cell 5: Ensemble + stacking
!python -m ensemble.stacking_ensemble --target tg
!python -m ensemble.stacking_ensemble --target egc
!python -m ensemble.build_ensemble --target tg
!python -m ensemble.build_ensemble --target egc
print("Ensemble + stacking complete")


In [ ]:
# Cell 6: Submission selection and export
import pandas as pd
import numpy as np

# Compare stacking vs weighted ensemble CV scores
# Pick the best per-target and blend
sub_stack_tg = pd.read_csv("outputs/submissions/v19_tg_stacking.csv")
sub_stack_egc = pd.read_csv("outputs/submissions/v19_egc_stacking.csv")

# Final submission
tg_preds = sub_stack_tg["target"].values
egc_preds = sub_stack_egc["target"].values
ids = sub_stack_tg["id"].values

# Combine: first half tg, second half egc
all_preds = np.concatenate([tg_preds, egc_preds])
all_ids = np.concatenate([ids.astype(str) + "_tg", ids.astype(str) + "_egc"])

submission = pd.DataFrame({"id": all_ids, "target": all_preds})
submission.to_csv("submission.csv", index=False)
print(f"Submission saved: {len(submission)} rows, range=[{all_preds.min():.2f}, {all_preds.max():.2f}]")
